In [5]:
import pandas as pd
import re
from sqlalchemy import create_engine
from sqlalchemy import text
import urllib
import networkx as nximport osfrom dotenv import load_dotenvload_dotenv()

In [6]:
# CONEXIÓN A SQL SERVER

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [7]:
# CARGAR DATOS

query = """
SELECT 
    Cliente,
    Email,
    Celular,
    Nombres
FROM dbo.Clientes
"""

df = pd.read_sql(query, engine)

print("Registros cargados:", len(df))

Registros cargados: 400677


In [8]:
# LIMPIEZA DE DATOS

df['Email'] = df['Email'].str.lower().str.strip()
df['Celular'] = df['Celular'].astype(str).str.strip()

# Ignorar valores basura
df.loc[df['Email'].isin(['negado@provenzal.net','none','']), 'Email'] = None
df.loc[df['Celular'].isin(['1111111111','nan','']), 'Celular'] = None

In [9]:
# CREAR GRAFO

G = nx.Graph()

# Agregar nodos
for cliente in df['Cliente']:
    G.add_node(cliente)

# Agrupar por Email
email_groups = df.groupby('Email')['Cliente'].apply(list)

for email, clientes in email_groups.items():
    if email and len(clientes) > 1:
        for i in range(len(clientes)-1):
            G.add_edge(clientes[i], clientes[i+1])

# Agrupar por Celular
cel_groups = df.groupby('Celular')['Cliente'].apply(list)

for cel, clientes in cel_groups.items():
    if cel and len(clientes) > 1:
        for i in range(len(clientes)-1):
            G.add_edge(clientes[i], clientes[i+1])

In [ ]:
# CLUSTERS = CLIENTE REAL

componentes = list(nx.connected_components(G))

map_cliente = {}

for i, comp in enumerate(componentes):
    id_unico = i + 100000   # iniciar desde 100000 
    for cliente in comp:
        map_cliente[cliente] = id_unico

df['ID_Cliente_Unico'] = df['Cliente'].map(map_cliente)

print("Clientes únicos reales:", len(componentes))

Clientes únicos reales: 381553


In [11]:
print(df.head())

     Cliente                     Email     Celular        Nombres  \
0          C  wandablcbogota@gmail.com  3022297475          WANDA   
1         C-   sp.blaschke@t-online.de        None        NATALIE   
2         C.    camilo.olaya@gmail.com        None         CAMILO   
3        C..   elenramirez@hotmail.com        None  ELENA RAMIREZ   
4  C:/242076        lowaco@hotmail.com        None        LORENZA   

   ID_Cliente_Unico  
0            100000  
1            100001  
2            100002  
3            100003  
4            100004  


In [12]:
df.to_excel("Clientes_Unicos.xlsx", index=False)